# K305 Week 1 Exploratory Data Analysis

**Topic 4 — Pedestrians and Cyclists at the Intersection (When and Where?)**  
Intersection K305, Dr.-Willi-Reiland-Ring × Würzburger Straße, Aschaffenburg

This notebook converts the Week 1 command-line EDA into a reproducible, visual analysis. It reports what the raw LiDAR object-list data supports and clearly separates frame-level detections from real-world people or trajectories.


## Dataset and interpretation

- The dataset contains 288 JSON files for 14 October 2026. Each represents a five-minute interval, with nominal 10 Hz recording (`Δt = 0.1 s`).
- Each frame contains object detections with a class, `track_id`, 3D position, yaw, dimensions, and horizontal velocity. Coordinates are relative to the roadside LiDAR sensor.
- A **detection** is one object observed in one frame; it is not a unique person or vehicle. A `track_id` is also not a reliable person count because IDs can fragment while a road user waits or becomes occluded.
- This is Week 1 EDA only. Cleaned trajectories, zones, 15-minute occupancy, route clusters, and VRU–vehicle conflicts are intentionally future work.


## Setup

This notebook uses Python's standard library for loading and analysis and Matplotlib for charts. If your environment does not already have a Jupyter kernel and Matplotlib, run the following in a terminal, then restart the kernel:

```bash
python -m pip install jupyter matplotlib
```

The saved outputs below were produced with the 12-file quick scan. Set `QUICK_MODE = False` only when you want to scan all 288 files.


In [1]:
from __future__ import annotations

import glob
import json
import math
import random
import statistics
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt

DATA_DIR = Path('database/k305_object_list_2026-10-14_aggregation_5min')
QUICK_MODE = True
VRU_CLASSES = {'Pedestrian', 'Cyclist'}
DELTA_T = 0.1
SAMPLE_SIZE = 50_000
RANDOM_SEED = 305

all_files = sorted(DATA_DIR.glob('*.json'))
assert all_files, f'No JSON files found in {DATA_DIR.resolve()}'
if QUICK_MODE:
    step = max(1, len(all_files) // 12)
    selected_files = all_files[::step][:12]
    demo_file = next(path for path in all_files if '08-00_to_08-04' in path.name)
    if demo_file not in selected_files:
        selected_files.append(demo_file)
else:
    selected_files = all_files

print(f'Discovered {len(all_files)} files; analysing {len(selected_files)} file(s).')


Discovered 288 files; analysing 12 file(s).


## Streaming analysis helpers

The scan processes one JSON file at a time. Coordinate ranges are updated incrementally, and speed/yaw distributions use reproducible reservoir samples; this avoids retaining the full multi-gigabyte dataset and avoids taking all distribution samples from the first files only.


In [2]:
def reservoir_add(sample, seen, value, rng):
    seen += 1
    if len(sample) < SAMPLE_SIZE:
        sample.append(value)
    else:
        index = rng.randrange(seen)
        if index < SAMPLE_SIZE:
            sample[index] = value
    return seen

def scan_files(files):
    counts, raw_ids = defaultdict(int), defaultdict(set)
    hourly_total, hourly_vru = defaultdict(int), defaultdict(int)
    ranges = {axis: [math.inf, -math.inf] for axis in 'xyz'}
    speeds, speed_seen, yaw, yaw_seen = defaultdict(list), defaultdict(int), [], 0
    segment_total = segment_short = frames_total = detection_total = null_groups = 0
    demo = {'Pedestrian': defaultdict(int), 'Cyclist': defaultdict(int)}
    demo_file = next(path for path in files if '08-00_to_08-04' in path.name)
    rng = random.Random(RANDOM_SEED)

    for path in files:
        with path.open(encoding='utf-8') as handle:
            payload = json.load(handle)
        local_track_frames = defaultdict(int)
        for frame in payload.get('frames', []):
            frames_total += 1
            objects = frame.get('objects', [])
            try:
                hour = datetime.strptime(frame.get('timestamp_local', '')[:16], '%Y-%m-%dT%H:%M').hour
            except ValueError:
                hour = -1
            hourly_total[hour] += len(objects)
            for obj in objects:
                detection_total += 1
                cls, tid = obj.get('class_name', 'Unknown'), obj.get('track_id', -1)
                counts[cls] += 1; raw_ids[cls].add(tid); local_track_frames[tid] += 1
                if cls in VRU_CLASSES:
                    hourly_vru[hour] += 1
                    if path == demo_file:
                        demo[cls][tid] += 1
                box = obj.get('bounding_box', {})
                position, velocity = box.get('position', {}), obj.get('velocity', {})
                xyz = [position.get(axis) for axis in 'xyz']
                if any(value is None for value in xyz):
                    null_groups += 1
                else:
                    for axis, value in zip('xyz', xyz):
                        ranges[axis][0] = min(ranges[axis][0], value)
                        ranges[axis][1] = max(ranges[axis][1], value)
                vx, vy = velocity.get('x'), velocity.get('y')
                if vx is None or vy is None:
                    null_groups += 1; continue
                speed = math.hypot(vx, vy)
                speed_seen[cls] = reservoir_add(speeds[cls], speed_seen[cls], speed, rng)
                if cls in VRU_CLASSES and speed > 0.1 and box.get('yaw') is not None:
                    difference = abs(math.atan2(vy, vx) - box['yaw']) % (2 * math.pi)
                    yaw_seen = reservoir_add(yaw, yaw_seen, min(difference, 2 * math.pi - difference), rng)
        segment_total += len(local_track_frames)
        segment_short += sum(n <= 3 for n in local_track_frames.values())

    return {'counts': dict(counts), 'raw_ids': {k: len(v) for k, v in raw_ids.items()},
            'hourly_total': dict(hourly_total), 'hourly_vru': dict(hourly_vru),
            'ranges': ranges, 'speeds': dict(speeds), 'yaw': yaw, 'null_groups': null_groups,
            'segments': segment_total, 'short_segments': segment_short, 'frames': frames_total,
            'detections': detection_total, 'demo': {k: dict(v) for k, v in demo.items()}}

results = scan_files(selected_files)
print(f"Processed {results['frames']:,} frames and {results['detections']:,} frame-level detections.")


Processed 33,791 frames and 483,260 frame-level detections.


## 1. Dataset scale and class composition

These are **frame-level detection counts**. The raw-ID column is diagnostic only: it is not a count of unique people or vehicles.


In [3]:
total = results['detections']
rows = []
for cls, count in sorted(results['counts'].items(), key=lambda item: item[1], reverse=True):
    rows.append((cls, count, 100 * count / total, results['raw_ids'][cls]))

print(f"Files: {len(selected_files)} | Frames: {results['frames']:,} | Detections: {total:,}")
for cls, count, share, ids in rows:
    print(f'{cls:12} {count:>10,}  {share:>6.2f}%  {ids:>7,} raw IDs')

plt.figure(figsize=(8, 4))
plt.bar([row[0] for row in rows], [row[1] for row in rows], color='#2563eb')
plt.ylabel('Frame-level detections')
plt.title('Quick-scan class composition')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()


Class,Frame-level detections,Share,Distinct raw IDs (diagnostic)
Car,"344,277",71.24%,"5,336 raw IDs"
Pedestrian,"84,633",17.51%,"2,366 raw IDs"
Van,"23,661",4.90%,933 raw IDs
Cyclist,"14,964",3.10%,683 raw IDs
Truck,"11,953",2.47%,419 raw IDs
Motorcycle,"2,697",0.56%,91 raw IDs
Bus,"1,075",0.22%,83 raw IDs


Rendered table/chart

## 2. Physical validation

Position ranges check the observed sensor footprint. Speed statistics are reproducible reservoir samples across the selected files, not a complete trajectory-speed analysis.


In [4]:
for axis in 'xyz':
    lower, upper = results['ranges'][axis]
    print(f'{axis.upper()}: {lower:+.2f} to {upper:+.2f} m (span {upper-lower:.1f} m)')

for cls in ['Pedestrian', 'Cyclist', 'Car', 'Van', 'Truck']:
    values = sorted(results['speeds'].get(cls, []))
    if values:
        p95 = values[int(.95 * (len(values) - 1))]
        print(f'{cls:12}: n={len(values):,}, median={statistics.median(values):.2f}, p95={p95:.2f}, max={max(values):.2f} m/s')

plt.figure(figsize=(8, 4))
for cls, color in [('Pedestrian', '#2563eb'), ('Cyclist', '#f97316')]:
    plt.hist(results['speeds'][cls], bins=40, alpha=.55, label=cls, color=color)
plt.xlim(0, 8); plt.xlabel('Speed (m/s)'); plt.ylabel('Sampled detections')
plt.title('VRU speed distributions (reservoir sample)'); plt.legend(); plt.tight_layout(); plt.show()


Class,Sample n,Median m/s,95th percentile m/s,Maximum m/s
Pedestrian,"50,000",0.16,1.64,5.10
Cyclist,"14,964",0.34,5.10,8.96
Car,"50,000",0.36,8.79,15.55
Van,"23,661",0.43,8.13,13.96
Truck,"11,953",0.65,7.19,11.85


Rendered table/chart

### Yaw check: calibration required

The raw comparison of `atan2(vy, vx)` with bounding-box yaw has a median absolute difference of **84.8°** (95th percentile: **174.9°**) in the quick scan. This contradicts the earlier 1.9° alignment claim. Therefore, this notebook does **not** treat raw yaw as a validated travel-direction proxy. Its coordinate convention must be calibrated before it is used for route or conflict analysis.


In [5]:
yaw_degrees = [math.degrees(value) for value in results['yaw']]
print(f"Yaw comparison sample: {len(yaw_degrees):,} moving VRU detections")
print(f"Median absolute heading–yaw difference: {statistics.median(yaw_degrees):.1f}°")
print(f"95th percentile: {sorted(yaw_degrees)[int(.95*(len(yaw_degrees)-1))]:.1f}°")
plt.figure(figsize=(8, 3.5)); plt.hist(yaw_degrees, bins=36, color='#dc2626')
plt.xlabel('Absolute heading–yaw difference (degrees)'); plt.ylabel('Sampled detections')
plt.title('Raw yaw is not yet calibrated as a direction proxy'); plt.tight_layout(); plt.show()


Yaw comparison sample: 50,000 moving VRU detections
Median absolute heading–yaw difference: 84.8°
95th percentile: 174.9°


## 3. Data cleanliness

No incomplete coordinate or velocity groups were found in this quick scan. Short-track values below refer to **file-local `(track_id, file)` segments**. They are an EDA diagnostic, not final trajectories, because tracks can cross files and IDs can change.


In [6]:
short_share = 100 * results['short_segments'] / results['segments']
print(f"Incomplete position/velocity groups: {results['null_groups']:,}")
print(f"File-local track segments: {results['segments']:,}")
print(f"Short segments (≤3 frames / ≤0.3 s): {results['short_segments']:,} ({short_share:.2f}%)")
print('Proposed later cleaning rule: filter tracks shorter than 5 frames, then evaluate cross-file continuity and ID stitching.')


Incomplete position/velocity groups: 0
File-local track segments: 9,885
Short segments (≤3 frames / ≤0.3 s): 1,458 (14.75%)
Proposed later cleaning rule: filter tracks shorter than 5 frames, then evaluate cross-file continuity and ID stitching.


## 4. Duration versus count

A person waiting at a signal is detected in many more frames than a person who crosses immediately. Frame counts therefore measure **observed presence**, not people. The table below uses the 08:00–08:04 file and reports raw track-segment durations; it is **not** spatial-zone dwell time because no crosswalk polygon has been defined yet.


In [7]:
duration_rows = []
for cls in ('Pedestrian', 'Cyclist'):
    durations = [frames * DELTA_T for frames in results['demo'][cls].values()]
    duration_rows.append((cls, len(durations), statistics.median(durations), statistics.mean(durations), max(durations), sum(durations)))
    print(f'{cls}: {len(durations)} raw tracks; median observed duration = {statistics.median(durations):.1f} s; total observed duration = {sum(durations):.1f} s')

plt.figure(figsize=(7, 4))
plt.bar([row[0] for row in duration_rows], [row[-1] for row in duration_rows], color=['#2563eb', '#f97316'])
plt.ylabel('Observed track-segment seconds')
plt.title('08:00–08:04 raw-track presence (not zone dwell time)')
plt.tight_layout(); plt.show()


Class,Raw tracks,Median seconds,Mean seconds,Longest seconds,Total observed seconds
Pedestrian,197,0.9,2.9,38.1,578.8
Cyclist,71,1.4,3.0,56.8,209.7


Rendered table/chart

## 5. Hourly quick-scan profile

This chart aggregates detections from the selected two-hourly files. It is a quick-scan diagnostic, not a continuous 24-hour hourly profile and not the required 15-minute occupancy curve.


In [8]:
hours = sorted(hour for hour in results['hourly_total'] if hour >= 0)
plt.figure(figsize=(10, 4))
plt.plot(hours, [results['hourly_total'][hour] for hour in hours], marker='o', label='All detections')
plt.plot(hours, [results['hourly_vru'][hour] for hour in hours], marker='o', label='VRU detections')
plt.xlabel('Hour of day'); plt.ylabel('Frame-level detections in selected file')
plt.title('Quick-scan temporal profile'); plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.show()


Rendered table/chart

## Week 2 handoff

Before answering “when and where?” at trajectory level, the next stage must: (1) filter short trajectories; (2) stitch compatible ID fragments and five-minute boundaries; (3) define or infer crosswalk, sidewalk, and cycling-zone geometry; (4) aggregate 15-minute zone occupancy in person-seconds/cyclist-seconds; (5) cluster cleaned VRU routes; and (6) calculate calibrated VRU–vehicle spatial overlap. Signal phase data and an authoritative intersection map remain valuable inputs.
